In [ ]:
from pathlib import Path
from atlas.io import get_valid_slice_folders

In [ ]:
#testing data path
series_folder = Path(
    "/home/xcamra/projects/data4test/Section Set 6"
)
print(series_folder.exists())

In [ ]:
from atlas.io import extract_s_number

series_list, failed_folders = get_valid_slice_folders(series_folder)
# Assign initial storage indices in numeric ATLAS section order.
series_list = sorted(series_list, key=extract_s_number)

In [ ]:
print(f"\nFound {len(series_list)} series.")

for f in series_list:
    print(f)

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

import json

import numpy as np
import tifffile as tiff
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import minimum_spanning_tree

from atlas.image_analysis import calculate_mask_roi, mask_low_and_saturation

from atlas.io import extract_s_number, get_metadata_for_stitched_tif
from atlas.stitching import (
    stitch_ATLAS_tiles,
    add_tile_overlap_columns,
    apply_transforms_and_stitch,
    build_adjacency_matrix_from_costs,
    build_transform_dict_from_mst,
    get_tiles_dataframe,
    match_tiles,
)

In [ ]:
from atlas.io import StitchedZarrStore

# Keep this trial separate from the existing sequential TIFFs and Zarr store.
sequential_reference_folder = series_folder / "older-sequential"
assert sequential_reference_folder.is_dir(), sequential_reference_folder
parallel_output_folder = series_folder / "parallel_worker_test"
parallel_output_folder.mkdir(exist_ok=True)
store = StitchedZarrStore.open(parallel_output_folder / "stitched_sections.zarr")

# Start with a few sections; use None to process all valid sections.
SECTION_NUMBERS_TO_PROCESS = None#[1, 2, 3]
MAX_WORKERS = 4
buffer_in_microns = 1
max_shift_in_pixles = 200

if SECTION_NUMBERS_TO_PROCESS is None:
    selected_series = series_list
else:
    selected_numbers = set(SECTION_NUMBERS_TO_PROCESS)
    selected_series = [
        folder for folder in series_list
        if extract_s_number(folder) in selected_numbers
    ]

jobs = []
for raw_data_folder in selected_series:
    mif_files = [
        file for file in raw_data_folder.iterdir()
        if file.is_file() and file.suffix.lower() == ".ve-mif"
    ]
    if len(mif_files) != 1:
        print(f"Skipping {raw_data_folder}: expected one .ve-mif file, found {len(mif_files)}")
        continue

    mif_file = mif_files[0]
    mif_metadict = get_metadata_for_stitched_tif(mif_file)
    first_tif_path = next(raw_data_folder.glob("*.tif"))
    filename_s_number = extract_s_number(first_tif_path)
    output_tif_path = parallel_output_folder / f"stitched_image_S_{filename_s_number}.tiff"
    output_csv_path = parallel_output_folder / f"phaseCC_stitching_S_{filename_s_number}.csv"
    output_transforms_path = parallel_output_folder / f"transforms_S_{filename_s_number}.json"
    output_metadata_path = parallel_output_folder / f"stitched_image_S_{filename_s_number}_metadata.json"

    jobs.append({
        "series_folder": raw_data_folder,
        "mif_file": mif_file,
        "acquisition_metadata": mif_metadict,
        "filename_s_number": filename_s_number,
        "output_tif_path": output_tif_path,
        "output_csv_path": output_csv_path,
        "output_transforms_path": output_transforms_path,
        "output_metadata_path": output_metadata_path,
    })


def stitch_and_save(job, parallel_writer):
    """Stitch one section, save its TIFF companions, and submit it to Zarr."""
    stitched_img, tile_df, transform_dict, stitching_metadata = stitch_ATLAS_tiles(
        job["mif_file"],
        buffer_microns=buffer_in_microns,
        max_shift_pixels=max_shift_in_pixles,
    )
    oriented_img = np.flipud(stitched_img)
    json_ready_transforms = {key: value.tolist() for key, value in transform_dict.items()}

    tiff.imwrite(
        job["output_tif_path"],
        oriented_img,
        metadata={"axes": "YX", "AcquisitionMetadata": job["acquisition_metadata"]},
    )
    with open(job["output_metadata_path"], "w") as metadata_file:
        json.dump(stitching_metadata, metadata_file, indent=4)
    tile_df.to_csv(job["output_csv_path"], index=False)
    with open(job["output_transforms_path"], "w") as transforms_file:
        json.dump(json_ready_transforms, transforms_file, indent=2)

    z_index = parallel_writer.write_section(
        atlas_section_number=job["acquisition_metadata"]["atlas_section_number"],
        image=oriented_img,
        acquisition_metadata=job["acquisition_metadata"],
        stitching_metadata=stitching_metadata,
        transforms=json_ready_transforms,
        source_folder=job["series_folder"],
        tile_dataframe=tile_df,
    )
    return job["acquisition_metadata"]["atlas_section_number"], z_index


section_order = [job["acquisition_metadata"]["atlas_section_number"] for job in jobs]
if jobs:
    with store.parallel_writer(section_order) as parallel_writer:
        active_jobs = []
        for job in jobs:
            section_number = job["acquisition_metadata"]["atlas_section_number"]
            if parallel_writer.is_complete(section_number):
                print(f"Skipping S_{section_number}: already complete in Zarr.")
            else:
                active_jobs.append(job)

        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            futures = {
                executor.submit(stitch_and_save, job, parallel_writer): job
                for job in active_jobs
            }
            for future in as_completed(futures):
                job = futures[future]
                section_number = job["acquisition_metadata"]["atlas_section_number"]
                try:
                    completed_number, z_index = future.result()
                    print(f"Completed S_{completed_number} at z={z_index}.")
                except Exception as error:
                    parallel_writer.mark_failed(section_number, error)
                    print(f"Failed S_{section_number}: {error}")

# Compare the parallel result with the existing sequential TIFF outputs and
# companion files when those references are available.
for job in jobs:
    section_number = job["acquisition_metadata"]["atlas_section_number"]
    record = dict(store.sections[f"S_{section_number}"].attrs)
    if record.get("status") != "complete":
        continue

    reference_tif = sequential_reference_folder / f"stitched_image_S_{job['filename_s_number']}.tiff"
    if not reference_tif.exists():
        print(f"No sequential TIFF reference for S_{section_number}; skipping comparison.")
        continue

    parallel_image = store.read_section(section_number)
    reference_image = tiff.imread(reference_tif)
    np.testing.assert_array_equal(reference_image, parallel_image)
    with tiff.TiffFile(reference_tif) as reference_file:
        reference_acquisition = reference_file.shaped_metadata[0]["AcquisitionMetadata"]
    assert record["acquisition_metadata"] == reference_acquisition

    reference_metadata = sequential_reference_folder / f"stitched_image_S_{job['filename_s_number']}_metadata.json"
    reference_transforms = sequential_reference_folder / f"transforms_S_{job['filename_s_number']}.json"
    reference_csv = sequential_reference_folder / f"phaseCC_stitching_S_{job['filename_s_number']}.csv"
    if reference_metadata.exists():
        with open(reference_metadata) as metadata_file:
            assert record["stitching_metadata"] == json.load(metadata_file)
    if reference_transforms.exists():
        with open(reference_transforms) as transforms_file:
            assert record["transforms"] == json.load(transforms_file)
    if reference_csv.exists():
        assert record["tile_table_csv"] == reference_csv.read_text()
    print(f"S_{section_number}: TIFF and Zarr pixels and available metadata match.")


In [ ]:
import tifffile as tiff

In [ ]:
with tiff.TiffFile(sequential_reference_folder.joinpath("stitched_image_S_2.tiff")) as tif:
    metadata = tif.shaped_metadata[0]["AcquisitionMetadata"]

for m in metadata.items():
    print(m)

## Read a stitched section from the Zarr store

Use the ATLAS section number (the `S` number) to retrieve its image and stored records. The section metadata gives the corresponding Z index in the shared `(z, y, x)` image array.

In [ ]:
from io import StringIO
import pandas as pd

# Select an ATLAS section number, for example S_2.
atlas_section_number = 2
section_key = f"S_{atlas_section_number}"
section_record = dict(store.sections[section_key].attrs)
z_index = section_record["z_index"]

# Read the image cropped to this section's original stitched shape.
section_image = store.read_section(atlas_section_number)
print(f"{section_key} is stored at z={z_index}")
print(f"Image shape: {section_image.shape}, dtype: {section_image.dtype}")

# Acquisition metadata comes from the TIFF; these records replace its
# metadata JSON and transforms JSON companion files.
acquisition_metadata = section_record["acquisition_metadata"]
stitching_metadata = section_record["stitching_metadata"]
transforms = section_record["transforms"]

# The former phaseCC_stitching CSV is stored as text and read as a DataFrame.
tile_table_csv = section_record["tile_table_csv"]
tile_df = pd.read_csv(StringIO(tile_table_csv)) if tile_table_csv else None

print("Acquisition metadata fields:", list(acquisition_metadata))
print("Stitching metadata fields:", list(stitching_metadata or {}))
print("Transform entries:", len(transforms or {}))
print("Tile table shape:", tile_df.shape if tile_df is not None else None)